## 1장 1강 : LLM 애플리케이션의 입력과 출력 구조

### 의존 패키지 설치
```
uv add ipykernel langchain langchain-core langchain-ollama python-dotenv
```

### 3. LCEL 파이프라인 실습

#### 3.1 환경 변수 로드 및 패키지 불러오기

In [5]:
from langchain_ollama import ChatOllama 
from langchain_core.prompts import ChatPromptTemplate


#### 3.2 ChatPromptTemplate으로 입력 템플릿 조립하기

시스템 역할과 사용자 질문 변수({user_question})를 담은 템플릿 생성<br>
system: "당신은 컴퓨터 기초 개념을 일상적인 사물에 빗대어 설명하는 교육 전문가입니다. 2문장 이내로 친절하게 설명하세요."<br>
user: "{user_question}"

In [8]:
# 역할, 상황, 제한조건, 예시 -> 시스템 메세지 : SystemMessage(...)
# user : 사용자 질의 : HumanMessage(...)
# assistant : AI의 답변 : AIMessage(...)
## 시스템의 질의는 고정이지만, 사용자의 질의는 항상 바뀔 수 있다! / 템플릿을 만든 것 = 프롬프트 템플릿

prompt_template = ChatPromptTemplate.from_messages([
    ("system", "당신은 컴퓨터 기초 개념을 일상적인 사물에 빗대어 설명하는 교육 전문가입니다. 2문장 이내로 친절하게 설명하세요."),
    ("user", "{user_question}"),
])

템플릿에 텍스트용 질문 데이터를 주입하여 결과를 확인

user_question: "프로그래밍에서 '변수'가 무엇인가요?"

In [9]:
sample_prompt = prompt_template.invoke({"user_question": "프로그래밍에서 '변수'란 무엇인가요?"})

sample_prompt

ChatPromptValue(messages=[SystemMessage(content='당신은 컴퓨터 기초 개념을 일상적인 사물에 빗대어 설명하는 교육 전문가입니다. 2문장 이내로 친절하게 설명하세요.', additional_kwargs={}, response_metadata={}), HumanMessage(content="프로그래밍에서 '변수'란 무엇인가요?", additional_kwargs={}, response_metadata={})])

#### 3.3 ChatOllama로 mistral 모델 호출하기

ChatOllama 모델 인스턴스 생성

In [10]:
model = ChatOllama(
    model="mistral", 
    base_url="http://localhost:11434/") # Ollama API 서버 주소

model

ChatOllama(metadata={'lc_versions': {'langchain-core': '1.6.2', 'langchain': '1.4.0'}}, model='mistral', base_url='http://localhost:11434/')

앞서 만든 sample_prompt를 모델에 직접 전달하여 실행

In [11]:
response = model.invoke(sample_prompt)

response

AIMessage(content=' 변수는 컴퓨터 프로그램에서 데이터를 저장하고 조작하는 컨테이너입니다. 예를 들어, 학생들의 학점을 관리하는 프로그램에서, 학생의 이름과 점수를 따로 기억하는 것처럼, 프로그램은 변수에 이러한 데이터를 담아둘 수 있습니다. 변수는 이름을 가지고 있고, 이름을 통해 변수에 저장된 데이터를 쉽게 찾고 조작할 수 있습니다.', additional_kwargs={}, response_metadata={'model': 'mistral', 'created_at': '2026-09-09T05:45:55.4594279Z', 'done': True, 'done_reason': 'stop', 'total_duration': 7289897600, 'load_duration': 4148042400, 'prompt_eval_count': 108, 'prompt_eval_duration': 121147000, 'eval_count': 191, 'eval_duration': 3010165000, 'logprobs': None, 'model_name': 'mistral', 'model_provider': 'ollama'}, id='lc_run--01a084b3-6b85-7892-ae38-2d9d428b8d10-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 108, 'output_tokens': 191, 'total_tokens': 299})

#### 3.4 StrOutputParser로 순수 텍스트만 추출하기

문자열 출력 파서 생성

In [12]:
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

raw_response 객체에서 순수 텍스트만 추출

In [13]:
text = parser.invoke(response)

text    

' 변수는 컴퓨터 프로그램에서 데이터를 저장하고 조작하는 컨테이너입니다. 예를 들어, 학생들의 학점을 관리하는 프로그램에서, 학생의 이름과 점수를 따로 기억하는 것처럼, 프로그램은 변수에 이러한 데이터를 담아둘 수 있습니다. 변수는 이름을 가지고 있고, 이름을 통해 변수에 저장된 데이터를 쉽게 찾고 조작할 수 있습니다.'

#### 3.5 LCEL 파이프(|) 연산자로 완전한 체인 결합 및 실행하기

파이프(|) 연산자를 사용해 3개 컴포넌트를 하나의 파이프라인으로 연결

In [14]:
chain = prompt_template | model | parser

새로운 질문으로 파이프라인 전체 실행

In [15]:
res = chain.invoke({
    "user_question": "클래스(Class)에 대해서 알기 쉬운 비유를 통해 설명하세요."
    })

res

' 클래스(Class)는 일상생활에서 아동의 선물상자와 같습니다. 선물상자 안에는 다양한 선물들이 들어있습니다. 아동이 선물상자를 만들 때, 선물상자 자체와 그 안에 들어갈 선물들의 형태, 크기, 색상 등을 정하게 됩니다. 이것이 프로그래밍에서도 같습니다. 클래스는 프로그램에서 사용할 객체(선물)의 형태, 속성(색상, 크기 등), 행동(함수)을 정의합니다. 이렇게 만든 클래스를 통해 객체를 만들어 프로그램에 사용할 수 있습니다.'

### 4. 프롬프트 엔지니어링
#### 4.1 프롬프트 엔지니어링의 3대 핵심 역할
- 목표 명확화
- 제약 조건 부여
- 출력 규격 표준화

#### 4.2 AI 성능 확장 4단계 비교
- Prompt Engineering
- RAG (검색 증강 생성)
- Fine-tuning (미세 조정)
- AI Agent (에이전트)

In [16]:
#------실습-----

from langchain_ollama import ChatOllama 
from langchain_core.prompts import ChatPromptTemplate


In [17]:


prompt_template = ChatPromptTemplate.from_messages([
    ("system", "당신은 친절한 프리랜서 영어 강사입니다 가장 자연스러운 영어 표현 1개와 그 이유를 설명해 주세요."),
    ("user", "{user_question}"),
])

In [18]:
sample_prompt = prompt_template.invoke({"user_question": "남은 음식을 포장해 달라고 할 때 어떻게 말하나요?"})

sample_prompt

ChatPromptValue(messages=[SystemMessage(content='당신은 친절한 프리랜서 영어 강사입니다 가장 자연스러운 영어 표현 1개와 그 이유를 설명해 주세요.', additional_kwargs={}, response_metadata={}), HumanMessage(content='남은 음식을 포장해 달라고 할 때 어떻게 말하나요?', additional_kwargs={}, response_metadata={})])

In [19]:
model = ChatOllama(
    model="mistral", 
    base_url="http://localhost:11434/") # Ollama API 서버 주소

model

ChatOllama(metadata={'lc_versions': {'langchain-core': '1.6.2', 'langchain': '1.4.0'}}, model='mistral', base_url='http://localhost:11434/')

In [20]:
response = model.invoke(sample_prompt)

response

AIMessage(content=' 가장 자연스러운 영어 표현으로 "Could you wrap up the leftovers, please?" 이라는 말을 사용합니다. 이 말은 남은 음식을 포장하여 주신다는 의미로 사용되며, 친절하고 간결하면서도 요청의 의도가 명확합니다. 또한, "please"는 당사를 부녀로 부탁하는 것을 나타내며, 친절하고 존중하는 것에 어려움이 없습니다. 따라서 이 말은 매우 적절하며 일상적인 상황에서 자주 사용되는 표현입니다.', additional_kwargs={}, response_metadata={'model': 'mistral', 'created_at': '2026-09-09T06:46:16.5360671Z', 'done': True, 'done_reason': 'stop', 'total_duration': 6811438300, 'load_duration': 3368970700, 'prompt_eval_count': 95, 'prompt_eval_duration': 110121000, 'eval_count': 219, 'eval_duration': 3325759000, 'logprobs': None, 'model_name': 'mistral', 'model_provider': 'ollama'}, id='lc_run--01a084ea-ae36-7160-a4d9-51e82965271a-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 95, 'output_tokens': 219, 'total_tokens': 314})

In [21]:
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

In [22]:
chain = prompt_template | model | parser

In [23]:
res = chain.invoke({
    "user_question": "클래스(Class)에 대해서 알기 쉬운 비유를 통해 설명하세요."
    })

res

' 가장 자연스러운 영어 표현 1개는 "break the ice"입니다. 이 표현은 처음 만나고 대화하는 두 사람이 긍정적인 관계를 맺기 위해 겸손하고 친절한 말을 통해 눈물거리는 경계를 뛰어 넘기는 것을 의미합니다. 이 표현을 사용하면 두 사람 간의 거리를 줄이고 대화를 시작하기 때문에 소화되고 긍정적인 분위기를 만들어 줍니다.\n\n클래스(Class)에 대해서 알기 쉬운 비유는 "학교 반(Classroom)"입니다. 학교 반(Classroom)은 학생들이 수업을 받고 학습하는 공간입니다. 학생들이 같은 반에 있으면 같은 과목을 공부하고, 같은 시험을 보며, 같은 일을 합니다. 이 비유를 통해 학생들이 같은 학교에 같은 과목을 공부하는 것처럼, 같은 클래스(Class)에 가입하면 같은 언어를 공부하고, 같은 훈련을 받을 수 있습니다. 또한, 학생들이 같은 시험을 보고 같은 성적을 받을 수 있듯이, 같은 클래스(Class)에 있으면 같은 수준의 영어 능력을 보여줄 수 있습니다.'

====

In [38]:
import os
from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama
from langchain_core.output_parsers import StrOutputParser

In [32]:
load_dotenv()

True

In [33]:
# 키 로드 여부 검증
api_key = os.getenv("OPENAI_API_KEY")
if api_key:
    print("환경 변수 OPENAI_API_KEY가 성공적으로 로드되었습니다.")
else:
    print("경고: .env 파일에 OPENAI_API_KEY가 설정되어 있는지 확인하세요.")

환경 변수 OPENAI_API_KEY가 성공적으로 로드되었습니다.


In [43]:
prompt_template = ChatPromptTemplate.from_messages([
    ("system", " 필리핀 다바오 5일간의 4인 가족 여행 일정을 계획하려고 해."),
    ("user", "{user_question}")
])

In [44]:
sample_prompt = prompt_template.invoke({"user_question": "다바오의 맛집을 소개해줘."})

# 조립된 프롬프트 메시지 구조
print(sample_prompt)

messages=[SystemMessage(content=' 필리핀 다바오 5일간의 4인 가족 여행 일정을 계획하려고 해.', additional_kwargs={}, response_metadata={}), HumanMessage(content='다바오의 맛집을 소개해줘.', additional_kwargs={}, response_metadata={})]


In [45]:
# ChatOpenAI 모델 인스턴스 생성 (자동으로 환경 변수 OPENAI_API_KEY를 참조합니다)
model = ChatOllama(
    model="mistral"
)

In [48]:
# 앞서 만든 sample_prompt를 모델에 직접 전달하여 실행
raw_response = model.invoke(sample_prompt)


print(raw_response)

content=" 다바오(Davao)는 필리핀의 동부 지역 중심인 폴리시오(Davao City)의 주요 도시로, 여행가들에게 많은 인기를 얻고 있습니다. 다바오에서는 다양한 맛집이 분포되어 있으며, 맛집을 선택하는 데는 여러 가지 요인(예: 지역, 가격, 식사 시간)이 있습니다. 다음은 다바오에서 유명한 맛집 몇 가지입니다.\n\n1. Jack's Ridge Restaurant & Bar\n   - 위치: 다바오 시(Davao City), 1117 지역(Cagayan de Oro Road, Poblacion)\n   - 특징: 풍부한 뷰(폴리시오의 도시맵과 상징 탑)와 다양한 식품 선택 가능(예: 폴리시오 스타일 바베큐, 치즈 파스타, 파인애플 크루에스)\n   - 운영 시간: 11:00 AM - 10:00 PM(월 ~ 목), 11:00 AM - 11:00 PM(금 ~ 일)\n\n2. Matina Enclave Seafood Restaurant\n   - 위치: 다바오 시(Davao City), 8000 지역(Matina Aplaya, Matina)\n   - 특징: 다양한 해산물 선택 가능(예: 크레베이트, 샤시, 삼겹살)과 침족감 있는 양념\n   - 운영 시간: 11:00 AM - 10:00 PM(월 ~ 일)\n\n3. Rustic Kitchen + Bar\n   - 위치: 다바오 시(Davao City), 8000 지역(G/F Marco Polo Davao, Claro M. Recto Avenue)\n   - 특징: 기본 요리(예: 파스타, 샐데이, 파운더셔널)와 독일 요리 선택 가능\n   - 운영 시간: 11:00 AM - 11:00 PM(월 ~ 일)\n\n4. Oche Fusion & Tapas Bar\n   - 위치: 다바오 시(Davao City), 8000 지역(Flores Street corner M.H. del Pilar Street, Poblacion)\n   - 특징: 다양한 음식 선택 가능(예: 핫 포킹, 스테이크, 샌드위치)과 친절

In [49]:
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

In [50]:
chain = prompt_template | model | parser

In [51]:
res = chain.invoke({
    "user_question": "필리핀 여행일정을 짜주세요."
    })

res

' 다음은 5일간 4인 가족의 필리핀 다바오 여행 일정입니다.\n\n1. 일요일 - 도착 및 체크인\n   - 아침 : 도착 후 체크인 및 휴식\n   - 점심 : 다바오 중앙 시장(Davao City Public Market)에서 탄수당 높은 칼로리 강식 음식 먹기\n   - 오후 : 농장 투어(Fruit Farm Tour) 예약\n   - 저녁 : 다바오 중앙 시장에서 농부들이 직접 만든 빵, 야채, 고기 등 다양한 음식 탐방\n\n2. 월요일 - 다바오 도시 탐방\n   - 아침 : 빼빼로 밥(Pancit) 먹기\n   - 점심 : 다바오 아카데미 공원(Davao Academy of Music and Arts in the Country - Davao Amanac Park)에서 다양한 음식 맛보기\n   - 오후 : 제주 산맥 공원(Mt. Apo Geopark) 투어\n   - 저녁 : 다바오 중앙 시장에서 농부들이 직접 만든 빵, 야채, 고기 등 다양한 음식 탐방\n\n3. 화요일 - 다바오 웰컴 센터 및 자연 지대 투어\n   - 아침 : 빼빼로 밥(Pancit) 먹기\n   - 점심 : 다바오 웰컴 센터(Davao Crocodile Park and Wildlife Center)에서 자연 생태 시간대 경험\n   - 오후 : 다바오 웰컴 센터 동물 원(Davao Crocodile Park Animal Farm)에서 동물 잡아 먹이기\n   - 저녁 : 다바오 웰컴 센터 식당에서 삼계탕 같은 필리핀 특산품 먹기\n\n4. 수요일 - 다바오 해안가 투어\n   - 아침 : 빼빼로 밥(Pancit) 먹기\n   - 점심 : 다바오 해안가에서 바닷물 가츰(Grilled Seafood) 먹기\n   - 오후 : 다바오 해안가 여행 용기(Davao Travel Boat)에서 잠깐 정상 휴식\n   - 저녁 : 다바오 해안가 레스토랑에서 바닷물 생선 등 다양한 물고기 먹기\n\n5. 금요일 - 체크아웃\n   - 아침 : 빼빼로 밥(Pancit) 먹기\n   - 점

====

In [52]:
prompt_template = ChatPromptTemplate.from_messages([
    ("system", " 너는 지금부터 컴퓨터 기초 개념을 일상적인 사물에 빗대어 설명하는 교육 전문가의 역할을 수행한다."),
    ("user", "{user_question}")
])

In [53]:
sample_prompt = prompt_template.invoke({"user_question": "데이터베이스가 무엇인지 냉장고에 빗대어 설명해줘."})

# 조립된 프롬프트 메시지 구조
print(sample_prompt)

messages=[SystemMessage(content=' 너는 지금부터 컴퓨터 기초 개념을 일상적인 사물에 빗대어 설명하는 교육 전문가의 역할을 수행한다.', additional_kwargs={}, response_metadata={}), HumanMessage(content='데이터베이스가 무엇인지 냉장고에 빗대어 설명해줘.', additional_kwargs={}, response_metadata={})]


In [54]:
# ChatOpenAI 모델 인스턴스 생성 (자동으로 환경 변수 OPENAI_API_KEY를 참조합니다)
model = ChatOllama(
    model="mistral"
)

In [55]:
# 앞서 만든 sample_prompt를 모델에 직접 전달하여 실행
raw_response = model.invoke(sample_prompt)


print(raw_response)

content=' 데이터베이스는 냉장고와 비슷합니다. 냉장고에는 여러 종류의 식자재가 들어있고, 각 식자재는 다양한 정보가 있습니다. 이름, 유통기한, 유통방법 등이 있을 수 있습니다. 냉장고는 이러한 식자재들을 한 곳에 모아 관리하는 것과 같습니다.\n\n데이터베이스도 비슷합니다. 데이터베이스에는 다양한 정보가 들어있고, 각 정보는 다양한 속성을 가집니다. 예를 들어, 회사에서 고객 정보를 데이터베이스에 저장하면, 이름, 연락처, 주소 등이 있을 수 있습니다. 데이터베이스는 이러한 정보들을 하나의 곳에 모아 관리하는 것입니다.\n\n더욱 복잡한 데이터베이스에서는 데이터베이스 내의 정보들을 다양한 방법으로 조회하고, 수정하고, 추가하고, 삭제할 수 있습니다. 예를 들어, 판매 정보를 저장하고 있는 데이터베이스에서는 특정 주문에 대한 정보를 조회할 수 있고, 주문을 추가하거나 변경하거나 삭제할 수 있습니다.\n\n또한, 데이터베이스는 여러 사람이 동시에 접근하여 정보를 조회하거나 수정하거나 추가하거나 삭제할 수 있는 공동 작업을 할 수 있는 환경을 제공합니다. 이러한 공동 작업을 수행하기 위해 데이터베이스는 일관성 있는 정보를 유지하는 것이 중요합니다. 예를 들어, 동시에 두 사람이 같은 정보를 수정하는 경우, 한 사람이 수정한 정보가 유실되거나 꺠질 수 있기 때문입니다. 따라서 데이터베이스는 이러한 문제를 해결하기 위해 일관성을 유지하는 방법을 사용합니다.\n\n또한, 데이터베이스에는 다양한 타입의 정보를 저장할 수 있습니다. 예를 들어, 숫자, 문자, 날짜, 그림 등의 정보를 저장할 수 있습니다. 또한, 데이터베이스는 데이터를 저장하는 방식에도 다양하게 제공합니다. 예를 들어, 기본 데이터 타입(number, string, date 등)을 저장하는 방법과, 이미지 파일을 저장하는 방법, 비디오 파일을 저장하는 방법 등이 있습니다.\n\n마지막으로, 데이터베이스는 다양한 목적으로 사용할 수 있습니다. 예를 들어, 회사에서는 고객 정보를 관리하는 데이터베이스를 

In [56]:
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

In [57]:
chain = prompt_template | model | parser

In [58]:
res = chain.invoke({
    "user_question": "전문가야, 컴퓨터의 기초개념을 설명해봐."
    })

res

' 네, 즐겁게 설명해보도록 하겠습니다!\n\n1. **컴퓨터**: 컴퓨터는 기계의 집합으로, 입력을 받아 처리하여 출력을 내놓는 장치입니다. 예를 들어 '